# lawforge-harvest: Kimina-Prover-RL-1.7B pass@K proof harvester

Generates K diverse Lean 4 proof candidates per SAIR equational-theory problem (train + dev + hard2 + hard3 splits). Saves raw candidates to `/kaggle/working/harvested.jsonl` for local judging.

Runtime estimate (T4, K=32, 1417 problems): ~6-8h.

Output schema per row:
```json
{"id": str, "split": str, "eq1": str, "eq2": str, "label": str|null,
 "candidates": [str, ...]}
```

In [ ]:
!pip -q install 'vllm==0.6.4' 'transformers>=4.46' accelerate

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/lawforge'
if not os.path.isdir(REPO):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/PAMF2/lawforge.git', REPO])
sys.path.insert(0, REPO)
print('repo HEAD:',
      subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
from vllm import LLM, SamplingParams
import os

MODEL = os.environ.get('LAWFORGE_HARVEST_MODEL', 'AI-MO/Kimina-Prover-RL-1.7B')
K = int(os.environ.get('LAWFORGE_HARVEST_K', '32'))
MAX_TOKENS = int(os.environ.get('LAWFORGE_HARVEST_MAX_TOKENS', '2048'))
TEMP = float(os.environ.get('LAWFORGE_HARVEST_TEMP', '0.6'))
TOP_P = float(os.environ.get('LAWFORGE_HARVEST_TOP_P', '0.95'))

llm = LLM(model=MODEL, dtype='bfloat16', max_model_len=8192,
          gpu_memory_utilization=0.85, trust_remote_code=True)
params = SamplingParams(n=K, temperature=TEMP, top_p=TOP_P, max_tokens=MAX_TOKENS)
print(f'model={MODEL} K={K} temp={TEMP} top_p={TOP_P} max_tokens={MAX_TOKENS}')

In [ ]:
import json
from pathlib import Path

RAW_PROMPT = Path(f'{REPO}/solver/prompt_template.txt').read_text()
CHEATSHEET = Path(f'{REPO}/solver/cheatsheet.md').read_text()
PROMPT_TPL = RAW_PROMPT.replace('__CHEATSHEET__', CHEATSHEET)

INPUTS = Path(f'{REPO}/kaggle/harvest/inputs')
SPLITS = ['train_split', 'dev_split', 'hard2_test', 'hard3_test']

problems = []
for s in SPLITS:
    path = INPUTS / f'{s}.jsonl'
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            row['_split'] = s
            problems.append(row)
print(f'loaded {len(problems)} problems across {len(SPLITS)} splits')

In [ ]:
import re
_PH = re.compile(r'\{(problem|solver|history)\.[a-zA-Z_]+\}')

def fill(p: dict) -> str:
    eq1 = p.get('equation1') or p.get('hypothesis', '')
    eq2 = p.get('equation2') or p.get('goal', '')
    eq1_id = p.get('eq1_id', '')
    eq2_id = p.get('eq2_id', '')
    vars_ = {
        'problem.id': str(p.get('id', '')),
        'problem.eq1_id': str(eq1_id),
        'problem.eq2_id': str(eq2_id),
        'problem.equation1': eq1,
        'problem.equation2': eq2,
        'problem.equation1_id': f'Equation{eq1_id}',
        'problem.equation2_id': f'Equation{eq2_id}',
        'history.attempts': '(no prior attempts)',
        'history.round': '0',
        'history.last_error': '',
        'history.last_status': '',
        'solver.round': '0',
        'solver.stage': 'harvest',
        'solver.ce_hint': '',
    }
    out = PROMPT_TPL
    for k, v in vars_.items():
        out = out.replace('{' + k + '}', v)
    return _PH.sub('', out)

print('sample prompt for problem 0 (first 500 chars):')
print(fill(problems[0])[:500])

In [ ]:
import time
OUT = Path('/kaggle/working/harvested.jsonl')
BATCH = int(os.environ.get('LAWFORGE_HARVEST_BATCH', '8'))
TACTIC_KEYS = ('intro', 'rw', 'apply', 'have', 'exact', 'simp', 'aesop',
               'calc', 'refine', 'symm', 'cases', 'rfl', 'decide',
               'assumption', 'nth_rewrite', 'repeat', 'fun ')

def looks_like_proof(text: str) -> bool:
    s = text.strip()
    if len(s) < 4:
        return False
    low = s.lower()
    return any(k in low for k in TACTIC_KEYS)

t0 = time.time()
with OUT.open('w') as out:
    for i in range(0, len(problems), BATCH):
        batch = problems[i:i+BATCH]
        prompts = [fill(p) for p in batch]
        results = llm.generate(prompts, params, use_tqdm=False)
        for p, r in zip(batch, results):
            cands = [o.text for o in r.outputs if looks_like_proof(o.text)]
            out.write(json.dumps({
                'id': p.get('id', ''),
                'split': p.get('_split', ''),
                'eq1': p.get('equation1') or p.get('hypothesis', ''),
                'eq2': p.get('equation2') or p.get('goal', ''),
                'label': p.get('label'),
                'candidates': cands,
            }) + '\n')
            out.flush()
        done = i + len(batch)
        if done % (BATCH * 4) == 0 or done == len(problems):
            elapsed = time.time() - t0
            rate = done / max(1, elapsed)
            eta = (len(problems) - done) / max(0.01, rate)
            print(f'[{done}/{len(problems)}] {elapsed:.0f}s elapsed, '
                  f'{rate:.2f} prob/s, ETA {eta:.0f}s')
print('harvest done in', time.time() - t0, 's')
print('output bytes:', OUT.stat().st_size)

In [ ]:
!ls -lh /kaggle/working/harvested.jsonl
!wc -l /kaggle/working/harvested.jsonl
!head -1 /kaggle/working/harvested.jsonl | python3 -c 'import json,sys; r=json.loads(sys.stdin.read()); print("id=", r["id"], "candidates=", len(r["candidates"]), "first=", r["candidates"][0][:200] if r["candidates"] else "")'